In [ ]:
import pandas as pd  
from scan_spectrolyser import scan_helpers
from scan_spectrolyser import no3_calibrations
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MultipleLocator
from hampel import hampel
import os


# Data Processing

## Import all FP and Correct Timestamp

## Correct for Turbidity

In [ ]:
rme = scan_helpers.import_all_scan_fp('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Readings/', correct_dst=True, tz='US/Mountain')
rme = rme['2025-01-23':] # only use data after install date
rme_corrected = scan_helpers.correct_turbidity(rme)

## Apply Calibrations

In [ ]:
rme = scan_helpers.apply_calibrations(rme, [no3_calibrations.one_wavelength, no3_calibrations.two_wavelength, no3_calibrations.second_derivative], {no3_calibrations.second_derivative: {'output_calibrated':False}, no3_calibrations.two_wavelength: {'output_calibrated':False}, no3_calibrations.one_wavelength: {'output_calibrated':False}})
rme['two_wavelength_no3_mgl'] = (rme['two_wavelength'] * 3.6615-.0802)* 14/1000
rme['one_wavelength_no3_mgl'] = (rme['one_wavelength']*3.7314+.3704) * 14/1000
rme['second_derivative_no3_mgl'] = (rme['second_derivative'] * 42.7379-.2905)* 14/1000
rme['two_wavelength_no3_mgl_correct'] = (rme['two_wavelength'] * 5.2845-.07)* 14/1000
rme['one_wavelength_no3_mgl_correct'] = (rme['one_wavelength']*54.5596+.2994) * 14/1000
rme['second_derivative_no3_mgl_correct'] = (rme['second_derivative'] * 43.0776-.2904)* 14/1000

rme_corrected = scan_helpers.apply_calibrations(rme_corrected, [no3_calibrations.one_wavelength, no3_calibrations.two_wavelength, no3_calibrations.second_derivative], {no3_calibrations.second_derivative: {'output_calibrated':False}, no3_calibrations.two_wavelength: {'output_calibrated':False}, no3_calibrations.one_wavelength: {'output_calibrated':False}})
rme_corrected['two_wavelength_no3_mgl'] = (rme_corrected['two_wavelength'] * 3.6615-.0802)* 14/1000
rme_corrected['one_wavelength_no3_mgl'] = (rme_corrected['one_wavelength']*3.7314+.3704) * 14/1000
rme_corrected['second_derivative_no3_mgl'] = (rme_corrected['second_derivative'] * 42.7379-.2905)* 14/1000
rme_corrected['two_wavelength_no3_mgl_correct'] = (rme_corrected['two_wavelength'] * 5.2845-.07)* 14/1000
rme_corrected['one_wavelength_no3_mgl_correct'] = (rme_corrected['one_wavelength']*54.5596+.2994) * 14/1000
rme_corrected['second_derivative_no3_mgl_correct'] = (rme_corrected['second_derivative'] * 43.0776-.2904)* 14/1000
rme_corrected

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

rme_corrected.plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl','two_wavelength_no3_mgl_correct', 'second_derivative_no3_mgl_correct'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')
#rme.plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
ax.yaxis.set_major_locator(MultipleLocator(.5))

## Cleanup Bad Values From Site Visits

In [ ]:
fig, ax= plt.subplots()

rme_corrected[:'2025-01-23'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected[:'2025-01-23'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')


- loooks like the initail turbidity cleared up around 1-23-25 at 18:00
- second_derivative method  rejected the turbidity, returned to normal almost immediately

In [ ]:
fig, ax= plt.subplots()

rme_corrected['2025-01-30 10:00':'2025-01-30 13:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected['2025-01-30 10:00':'2025-01-30 13:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- no spike in corrected data

In [ ]:
fig, ax= plt.subplots()

rme_corrected['2025-02-12 10:20':'2025-02-13 00:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected['2025-02-12 10:20':'2025-02-13 00:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- looks like readings bad from 10:20 to 18:00
- again, second_derivative clears up before two wavelength method and absorbance

In [ ]:
fig, ax= plt.subplots()

rme_corrected['2025-02-25 10:30':'2025-02-25 15:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected['2025-02-25 10:30':'2025-02-25 15:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- data are bad from 2025-02-25 12:00 to 13:30.
- this time 2nd derivative and two_ewavelength recover about the same

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-03-11 08:00':'2025-03-11 15:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-03-11 08:00':'2025-03-11 15:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- data are bad from 9:00 to 13:00

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-03-25 10:00':'2025-03-25 14:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-03-25 10:00':'2025-03-25 14:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- data bad from 10:10 to 11:30

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-04-02 9:00':'2025-04-02 14:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-04-02 9:00':'2025-04-02 14:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

data bad from 9:30 to 10:30

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-04-17 10:00':'2025-04-17 16:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-04-17 10:00':'2025-04-17 16:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- no bad data from this field visit

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-05-01 10:00':'2025-05-01 16:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-05-01 10:00':'2025-05-01 16:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

- also no bad data here

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-05-12 10:00':'2025-05-12 16:00'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-05-12 10:00':'2025-05-12 16:00'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
fig, ax= plt.subplots()

rme_corrected.loc['2025-04-28'].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate', ax=ax, ylabel='Nitrate (mg/L)')
rme_corrected.loc['2025-04-28'].plot(y=[220.00, 265.00], ax=ax, secondary_y=True, ylabel='Absorbance')

In [ ]:
rme_bad = [
    ('2025-01-23 00:00', '2025-01-23 18:00'),
    ('2025-02-12 10:20', '2025-02-12 18:00'),
    ('2025-02-25 12:00', '2025-02-25 13:30'),
    ('2025-03-11 9:00', '2025-03-11 13:00'),
    ('2025-03-25 10:10', '2025-03-25 11:30'),
    ('2025-04-02 9:30', '2025-04-02 10:30'),
    ('2025-04-28 11:00', '2025-04-28 21:00'),

]
rme_mask = np.array([
    (rme.index >= start) & (rme.index <= end)
    for start, end in rme_bad
]).any(axis=0)

rme_cleaned = rme_corrected.loc[~rme_mask]
rme_cleaned

### Replotting the whole time series of cleaned values:

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

rme_cleaned.plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')
rme_cleaned.plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)
ax.yaxis.set_major_locator(MultipleLocator(.5))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8.5))

rme_cleaned['2/22/2025':].plot(y=['one_wavelength_no3_mgl', 'two_wavelength_no3_mgl', 'second_derivative_no3_mgl'], title = 'RME Calculated Nitrate Jan-Mar 2025', ax=ax, ylabel='Nitrate (mg/L)')
rme_cleaned['2/22/2025':].plot(y=[220.00, 265.00], secondary_y=True, ylabel='Absorbance', ax=ax)

## TODO:
- fill missing data from the 5/12-5/15 somehow
- figure out why you're getting negative values of two_wavelength_no3 

In [ ]:
rme_cleaned.to_csv('/Users/beneck/Library/CloudStorage/OneDrive-NortheasternUniversity/Boise Project/Data/Reynolds Nitrate Monitoring/SCAN Data/RME/Processed Data/rme_cleaned.csv')